# Propensity Score Matching Analysis for Klaatch Data

This notebook performs propensity score matching (PSM) to analyze CEL (emotional loneliness) scores across demographic groups. The analysis controls for confounding variables to enable fair comparisons between:
- Gender groups (Male vs Female)
- Racial groups (Black vs White)

The notebook also evaluates model fairness by testing prediction performance across subgroups.

## Setup & Configuration

In [ ]:
# Configuration Constants
DATA_FILE_PATH = "/home/vinmayk/Klaatch_final_data.csv"

# Database Configuration
DB_CONFIG = {
    'host': '127.0.0.1',
    'database': 'Audio_features',
    'username': '**************',
    'password': '**************'
}

# Database Table Names
TABLE_NAMES = {
    'male': 'stratified_male',
    'female': 'stratified_female',
    'black': 'stratified_black',
    'white': 'stratified_white'
}

# Analysis Parameters
RANDOM_STATE = 42
TRAIN_TEST_SPLIT = 0.8
N_ESTIMATORS = 100
MAX_ITER_LOGISTIC = 500

# Race Categories
WHITE_RACE_CATEGORIES = ['White', 'White, Jewish', 'White, Judaism', 'White,']
BLACK_RACE_CATEGORY = 'Black or African American'

## Imports

In [15]:
# Data manipulation and analysis
import numpy as np
import pandas as pd
import re

# Machine learning - preprocessing and models
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor

# Database connectivity
import mysql.connector
import sqlalchemy

## Function Definitions

In [16]:
def clean_text_column(df, text_column='Text'):
    """
    Clean text data by removing survey responses, timestamps, and noise words.
    
    Args:
        df (pd.DataFrame): DataFrame containing text data
        text_column (str): Name of the text column to clean
        
    Returns:
        pd.DataFrame: DataFrame with cleaned text column
    """
    patterns_to_remove = [
        'strongly agree', 'strongly disagree', 'strongly_agree', 'strongly_disagree',
        'agree\\w*', 'disagree\\w*', 'neutral', 'Speaker', 'strongly agree\\w*',
        'strongly disagree\\w*', 'strongly_agree\\w*', 'strongly_disagree\\w*',
        'Speaker\\w*', 'speaker\\w*', 'nn\\w*', 'nnspeaker\\w*',
        '!', '\\$', ',', '\\.', '\\.\\.', '\\.\\s*mm hmm', '\\.\\.',
        '0', '00 \\]', '0:00', '0:00:00',
        '1', '10', '100', '11', '12', '13', '14', '15', '16', '18',
        '2', '20', '20 years', '25', '3', '30', '4', '40',
        '5', '5 to 10', '50', '90',
        ':', ': 00 \\]', '\\?', '\\]', '\\] hello', '\\] hello \\.',
        "don't\\w*", 'not\\w*', 'strongly\\w*', 'definitely\\w*', 'neither\\w*', 'lisa\\w*'
    ]
    
    timestamp_pattern = r'\[\d{2}:\d{2}:\d{2}\]'
    
    escaped_patterns = []
    for pattern in patterns_to_remove:
        if '\\w' in pattern:
            escaped_patterns.append(pattern)
        else:
            escaped_patterns.append(re.escape(pattern))
    
    words_regex = r'\b(?:' + '|'.join(escaped_patterns) + r')\b'
    full_regex_pattern = f'(?:{words_regex})|(?:{timestamp_pattern})'
    
    df[text_column] = df[text_column].str.replace(
        full_regex_pattern, '', regex=True, flags=re.IGNORECASE
    )
    df[text_column] = df[text_column].str.replace(r'\s+', ' ', regex=True).str.strip()
    
    return df


def filter_and_encode_binary(df, comparison_type, white_categories, black_category):
    """
    Filter data and create binary encodings for propensity score matching.
    
    Args:
        df (pd.DataFrame): Input DataFrame
        comparison_type (str): Either 'gender' or 'race'
        white_categories (list): List of white race category strings
        black_category (str): Black race category string
        
    Returns:
        pd.DataFrame: Filtered and encoded DataFrame
    """
    df_copy = df.copy()
    
    if comparison_type == 'gender':
        df_copy = df_copy[df_copy['Gender'].isin(['Male', 'Female'])]
        df_copy = df_copy[~df_copy['Race'].isin(['Asian', 'White, Black or African American'])]
        df_copy['Race'] = df_copy['Race'].apply(
            lambda x: 1 if x in white_categories else 0
        )
    elif comparison_type == 'race':
        df_copy = df_copy[df_copy['Race'].isin(white_categories + [black_category])]
        df_copy['Gender'] = df_copy['Gender'].apply(
            lambda x: 1 if x == 'Female' else 0
        )
    
    return df_copy


def calculate_propensity_scores(df, target_col, feature_cols, max_iter=500):
    """
    Calculate propensity scores using logistic regression.
    
    Args:
        df (pd.DataFrame): Input DataFrame
        target_col (str): Name of target column (binary outcome)
        feature_cols (list): List of feature column names
        max_iter (int): Maximum iterations for logistic regression
        
    Returns:
        tuple: (DataFrame with propensity scores, trained model)
    """
    X = df[feature_cols]
    y = df[target_col]
    
    model = LogisticRegression(max_iter=max_iter)
    model.fit(X, y)
    
    propensity_scores = model.predict_proba(X)[:, 1]
    
    return propensity_scores, model


def perform_propensity_matching(treatment_df, control_df, propensity_col, random_state=42):
    """
    Perform 1:1 nearest neighbor propensity score matching.
    
    Args:
        treatment_df (pd.DataFrame): Treatment group DataFrame
        control_df (pd.DataFrame): Control group DataFrame
        propensity_col (str): Name of propensity score column
        random_state (int): Random seed for reproducibility
        
    Returns:
        pd.DataFrame: Balanced matched pairs dataset
    """
    matched_pairs = pd.DataFrame()
    control_pool = control_df.copy()
    
    for _, row in treatment_df.iterrows():
        closest_match = control_pool.iloc[
            (control_pool[propensity_col] - row[propensity_col]).abs().argsort()[:1]
        ]
        matched_pairs = pd.concat(
            [matched_pairs, row.to_frame().T, closest_match], 
            ignore_index=True
        )
        control_pool = control_pool.drop(closest_match.index)
    
    matched_pairs = matched_pairs.reset_index(drop=True)
    matched_pairs = matched_pairs.sample(frac=1, random_state=random_state).reset_index(drop=True)
    
    return matched_pairs


def connect_to_db(db_config):
    """
    Establish connection to MySQL database.
    
    Args:
        db_config (dict): Database configuration dictionary with keys:
                         'host', 'database', 'username', 'password'
                         
    Returns:
        mysql.connector.connection.MySQLConnection: Database connection object
        
    Raises:
        mysql.connector.Error: If connection fails
    """
    try:
        connection = mysql.connector.connect(
            host=db_config['host'],
            database=db_config['database'],
            user=db_config['username'],
            password=db_config['password']
        )
        return connection
    except mysql.connector.Error as err:
        print(f"Error connecting to database: {err}")
        raise


def create_table_if_not_exists(table_name, db_config):
    """
    Create database table if it does not exist.
    
    Args:
        table_name (str): Name of the table to create
        db_config (dict): Database configuration dictionary
        
    Raises:
        mysql.connector.Error: If table creation fails
    """
    create_table_query = f"""
    CREATE TABLE IF NOT EXISTS {table_name} (
        message_id VARCHAR(191) NOT NULL,
        message TEXT,
        Date DATE,
        CEL_Total INT,
        CELVAL1 INT,
        CELVAL2 INT,
        CELVAL3 INT,
        klaatch_id INT,
        PRIMARY KEY (message_id)
    );
    """
    
    try:
        connection = connect_to_db(db_config)
        cursor = connection.cursor()
        cursor.execute(create_table_query)
        connection.commit()
        print(f"Table {table_name} is ready.")
    except mysql.connector.Error as err:
        print(f"Error creating table {table_name}: {err}")
        raise
    finally:
        if cursor:
            cursor.close()
        if connection:
            connection.close()


def insert_data_to_db(df, table_name, db_config):
    """
    Insert DataFrame data into database table.
    
    Args:
        df (pd.DataFrame): DataFrame containing data to insert
        table_name (str): Name of the database table
        db_config (dict): Database configuration dictionary
        
    Raises:
        mysql.connector.Error: If insertion fails
    """
    create_table_if_not_exists(table_name, db_config)
    
    insert_query = f"""
    INSERT IGNORE INTO {table_name} (
        message_id, message, Date, CEL_Total, CELVAL1, CELVAL2, CELVAL3, klaatch_id
    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
    """
    
    data = df[[
        'Filename', 'Text', 'Date', 'CEL Total',
        'CELVAL1', 'CELVAL2', 'CELVAL3', 'Old ID'
    ]].values.tolist()
    
    try:
        connection = connect_to_db(db_config)
        cursor = connection.cursor()
        cursor.executemany(insert_query, data)
        connection.commit()
        print(f"Inserted {cursor.rowcount} rows successfully into table {table_name}.")
    except mysql.connector.Error as err:
        print(f"Error inserting data: {err}")
        raise
    finally:
        if cursor:
            cursor.close()
        if connection:
            connection.close()


def evaluate_subgroup_performance(df, text_col, target_col, id_col, train_size=0.8, 
                                 n_estimators=100, random_state=42):
    """
    Evaluate prediction model performance on a subgroup.
    
    Args:
        df (pd.DataFrame): Subgroup DataFrame
        text_col (str): Name of text column
        target_col (str): Name of target column (CEL Total)
        id_col (str): Name of ID column for splitting
        train_size (float): Proportion of data for training
        n_estimators (int): Number of trees in random forest
        random_state (int): Random seed
        
    Returns:
        dict: Dictionary containing model_score and mae
    """
    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(df[text_col])
    y = df[target_col]
    
    unique_ids = df[id_col].unique()
    rng = np.random.RandomState(random_state)
    rng.shuffle(unique_ids)
    
    train_count = int(len(unique_ids) * train_size)
    train_ids = unique_ids[:train_count]
    test_ids = unique_ids[train_count:]
    
    train_mask = df[id_col].isin(train_ids).to_numpy()
    test_mask = df[id_col].isin(test_ids).to_numpy()
    
    X_train, X_test = X[train_mask], X[test_mask]
    y_train = df[train_mask][target_col]
    y_test = df[test_mask][target_col]
    
    model = RandomForestRegressor(n_estimators=n_estimators, random_state=random_state)
    model.fit(X_train, y_train)
    
    predictions = model.predict(X_test)
    model_score = model.score(X_test, y_test)
    mae = np.abs(y_test - predictions).mean()
    
    return {
        'model_score': model_score,
        'mae': mae,
        'train_shape': X_train.shape,
        'test_shape': X_test.shape
    }

## Data Loading & Preprocessing

In [17]:
# Load data
try:
    df = pd.read_csv(DATA_FILE_PATH)
    print(f"Loaded data with shape: {df.shape}")
except FileNotFoundError:
    print(f"Error: File not found at {DATA_FILE_PATH}")
    raise
except Exception as e:
    print(f"Error loading data: {e}")
    raise

# Clean column names and drop unnecessary columns
if 'Unnamed: 0' in df.columns:
    df.drop('Unnamed: 0', axis=1, inplace=True)

df.rename(columns={'New ID_x': 'New ID', 'Old ID_x': 'Old ID'}, inplace=True)

Loaded data with shape: (1465, 29)


In [18]:
# Clean text data
df = clean_text_column(df, text_column='Text')
print("Text cleaning completed.")

Text cleaning completed.


In [19]:
# Select relevant columns and add word count
df = df[[
    'New ID', 'Old ID', 'Filename', 'Text', 'CEL Total', 
    'CELVAL1', 'CELVAL2', 'CELVAL3', 'Age', 'Gender', 'Race', 'Date'
]].copy()

df['word_count'] = df['Text'].apply(lambda x: len(str(x).split()))

print(f"Final preprocessed data shape: {df.shape}")
df.head()

Final preprocessed data shape: (1465, 13)


,New ID,Old ID,Filename,Text,CEL Total,CELVAL1,CELVAL2,CELVAL3,Age,Gender,Race,Date,word_count
0,NaN,559.0,559_2021-01-22,"Hello. OK. Sure. Yesterday, I went to an appoi...",5,1,0,4,NaN,NaN,NaN,2021-01-22,365
1,938730.0,66.0,66_2021-01-26,Hello. Please leave a message after the tone. ...,3,1,1,1,95.0,Female,White,2021-01-26,351
2,938936.0,340.0,340_2021-01-26,"Yes. I'm fine, how are you? Ok. Well, what I u...",2,1,0,1,82.0,Female,Black or African American,2021-01-26,904
3,NaN,343.0,343_2021-01-26,"Hello. All right. I'll do it now. Sure. No, no...",1,1,0,0,NaN,NaN,NaN,2021-01-26,352
4,938879.0,383.0,383_2021-01-27,"Hello. Hi, I'm. Ok. OK. Yes, okay. I wake up, ...",5,1,3,1,72.0,Female,White,2021-01-27,131


## Data Filtering and Binary Encoding

### Purpose
Prepare separate datasets for gender and race comparisons by:
1. Filtering out small/mixed groups (Asian, mixed-race participants)
2. Binary encoding demographic variables for propensity score models

### Encoding Strategy
- **For Gender Comparison**: Race encoded as 0 (Black) or 1 (White)
- **For Race Comparison**: Gender encoded as 0 (Male) or 1 (Female)

In [20]:
# Create gender comparison dataset
gender_data = filter_and_encode_binary(
    df, 
    comparison_type='gender',
    white_categories=WHITE_RACE_CATEGORIES,
    black_category=BLACK_RACE_CATEGORY
)

# Create race comparison dataset
race_data = filter_and_encode_binary(
    df,
    comparison_type='race',
    white_categories=WHITE_RACE_CATEGORIES,
    black_category=BLACK_RACE_CATEGORY
)

print(f"Gender comparison data shape: {gender_data.shape}")
print(f"Race comparison data shape: {race_data.shape}")

Gender comparison data shape: (1328, 13)
Race comparison data shape: (1307, 13)


## Propensity Score Calculation

### Gender Propensity Scores
Calculate probability of being Female given Race, Age, and CEL Total scores.

In [21]:
# Encode Gender as binary
gender_data['Gender'] = gender_data['Gender'].replace({'Male': 0, 'Female': 1}).astype(int)

# Calculate propensity scores
propensity_scores, gender_model = calculate_propensity_scores(
    gender_data,
    target_col='Gender',
    feature_cols=['Race', 'Age', 'CEL Total'],
    max_iter=MAX_ITER_LOGISTIC
)

gender_data['gender_propensity_score'] = propensity_scores
print("Gender propensity scores calculated.")

Gender propensity scores calculated.


/tmp/ipykernel_2796547/2333249005.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gender_data['Gender'] = gender_data['Gender'].replace({'Male': 0, 'Female': 1}).astype(int)


### Race Propensity Scores
Calculate probability of being White given Gender, Age, and CEL Total scores.

In [22]:
# Create binary Race column
race_data['Race_binary'] = race_data['Race'].apply(
    lambda x: 1 if x in WHITE_RACE_CATEGORIES else 0
).astype(int)

# Calculate propensity scores
propensity_scores, race_model = calculate_propensity_scores(
    race_data,
    target_col='Race_binary',
    feature_cols=['Gender', 'Age', 'CEL Total'],
    max_iter=MAX_ITER_LOGISTIC
)

race_data['race_propensity_score'] = propensity_scores
print("Race propensity scores calculated.")

Race propensity scores calculated.


## Matching Analysis

### Gender Matching
Perform 1:1 nearest neighbor matching to create balanced Male-Female comparison groups.

In [23]:
# Separate by gender
male = gender_data[gender_data['Gender'] == 0].copy()
female = gender_data[gender_data['Gender'] == 1].copy()

print(f"Male participants: {len(male)}")
print(f"Female participants: {len(female)}")

# Perform initial matching
matched_pairs = pd.DataFrame()
male_pool = male.copy()

for _, row in female.iterrows():
    closest_match = male_pool.iloc[
        (male_pool['gender_propensity_score'] - row['gender_propensity_score']).abs().argsort()[:1]
    ]
    matched_pairs = pd.concat(
        [matched_pairs, row.to_frame().T, closest_match], 
        ignore_index=True
    )
    male_pool = male_pool.drop(closest_match.index)

matched_pairs = matched_pairs.reset_index(drop=True)

# Balance the matched groups
female_matched = matched_pairs[matched_pairs['Gender'] == 1]
male_matched = matched_pairs[matched_pairs['Gender'] == 0]

min_entries = min(len(female_matched), len(male_matched))

female_final = female_matched.sort_values(by='gender_propensity_score').head(min_entries)
male_final = male_matched.sort_values(by='gender_propensity_score').head(min_entries)

final_gender = pd.concat([female_final, male_final]).reset_index(drop=True)
final_gender = final_gender.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f"\nMatched gender dataset shape: {final_gender.shape}")
print(f"Balanced Male: {len(male_final)}, Female: {len(female_final)}")
final_gender.head()

Male participants: 397
Female participants: 931

Matched gender dataset shape: (794, 14)
Balanced Male: 397, Female: 397


,New ID,Old ID,Filename,Text,CEL Total,CELVAL1,CELVAL2,CELVAL3,Age,Gender,Race,Date,word_count,gender_propensity_score
0,967643.0,835.0,835_2024-02-16,"000002] Hello. Okay. How are you doing, Amy? S...",1,0,0,1,73.0,0,0,2024-02-16,1279,0.734129
1,938955.0,360.0,360_2022-06-06,Fine. Are you? Yes. All right? Yeah. Fine. The...,1,0,0,1,87.0,0,0,2022-06-06,849,0.845351
2,938913.0,661.0,661_2022-06-22,Okay. Hi. Right. Yeah. Yes. I get to see my gr...,2,0,1,1,70.0,1,1,2022-06-22,530,0.552753
3,938896.0,755.0,755_2022-07-26,"Hi, Michelle. Okay. You said you said 120 days...",0,0,0,0,74.0,0,1,2022-07-26,1746,0.611864
4,938913.0,661.0,661_2023-05-15,"Hello. Hi. Good. Yeah. Oh, yeah. Yeah. Yeah. U...",2,1,0,1,70.0,1,1,2023-05-15,461,0.552753


### Race Matching
Perform 1:1 nearest neighbor matching to create balanced Black-White comparison groups.

In [24]:
# Separate by race
white = race_data[race_data['Race'].isin(WHITE_RACE_CATEGORIES)].copy()
black = race_data[race_data['Race'] == BLACK_RACE_CATEGORY].copy()

print(f"White participants: {len(white)}")
print(f"Black participants: {len(black)}")

# Perform initial matching
matched_pairs = pd.DataFrame()
black_pool = black.copy()

for _, row in white.iterrows():
    closest_match = black_pool.iloc[
        (black_pool['race_propensity_score'] - row['race_propensity_score']).abs().argsort()[:1]
    ]
    matched_pairs = pd.concat(
        [matched_pairs, row.to_frame().T, closest_match], 
        ignore_index=True
    )
    black_pool = black_pool.drop(closest_match.index)

matched_pairs = matched_pairs.reset_index(drop=True)

# Balance the matched groups
white_matched = matched_pairs[matched_pairs['Race'].isin(WHITE_RACE_CATEGORIES)]
black_matched = matched_pairs[matched_pairs['Race'] == BLACK_RACE_CATEGORY]

min_entries = min(len(white_matched), len(black_matched))

white_final = white_matched.sort_values(by='race_propensity_score').head(min_entries)
black_final = black_matched.sort_values(by='race_propensity_score').head(min_entries)

final_race = pd.concat([white_final, black_final]).reset_index(drop=True)
final_race = final_race.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f"\nMatched race dataset shape: {final_race.shape}")
print(f"Balanced White: {len(white_final)}, Black: {len(black_final)}")
final_race.head()

White participants: 924
Black participants: 383

Matched race dataset shape: (766, 15)
Balanced White: 383, Black: 383


,New ID,Old ID,Filename,Text,CEL Total,CELVAL1,CELVAL2,CELVAL3,Age,Gender,Race,Date,word_count,Race_binary,race_propensity_score
0,938682.0,545.0,545_2023-04-06,"Hello. Hello. Good morning. How are you? Uh, n...",3,1,1,1,65.0,1,Black or African American,2023-04-06,790,0,0.580088
1,938695.0,687.0,687_2022-08-08,"Hello. Yet. Oh, okay. Yeah. Yeah. How you doin...",0,0,0,0,74.0,0,Black or African American,2022-08-08,1436,0,0.738187
2,938698.0,170.0,938698_2024-10-30,Hello. I'm doing pretty well. Yeah. What am I ...,2,1,1,0,87.0,1,White,2024-10-30,1206,1,0.69502
3,938917.0,173.0,173_2022-11-10,"Hello.: Oh, hi. How are you?: Good, good, good...",0,0,0,0,90.0,1,Black or African American,2022-11-10,3206,0,0.670894
4,938673.0,144.0,144_2024-07-17,"ello? Oh, okay. Um, are you recording now? Alr...",3,1,1,1,83.0,1,White,2024-07-17,3554,1,0.692447


## Subgroup Analysis

### Model Fairness Evaluation
Test whether prediction models work equally well across demographic subgroups.
Lower MAE indicates better prediction accuracy.

In [25]:
# Prepare subgroup datasets
stratified_male = final_gender[final_gender['Gender'] == 0].copy()
stratified_female = final_gender[final_gender['Gender'] == 1].copy()
stratified_white = final_race[final_race['Race'].isin(WHITE_RACE_CATEGORIES)].copy()
stratified_black = final_race[final_race['Race'] == BLACK_RACE_CATEGORY].copy()

print(f"Stratified Male: {stratified_male.shape}")
print(f"Stratified Female: {stratified_female.shape}")
print(f"Stratified White: {stratified_white.shape}")
print(f"Stratified Black: {stratified_black.shape}")

Stratified Male: (397, 14)
Stratified Female: (397, 14)
Stratified White: (383, 15)
Stratified Black: (383, 15)


In [26]:
# Evaluate each subgroup
subgroups = {
    'Females': stratified_female,
    'Males': stratified_male,
    'Whites': stratified_white,
    'Blacks': stratified_black
}

results = {}
mae_values = []

for name, subgroup_df in subgroups.items():
    print(f"\nEvaluating {name}...")
    
    result = evaluate_subgroup_performance(
        df=subgroup_df,
        text_col='Text',
        target_col='CEL Total',
        id_col='New ID',
        train_size=TRAIN_TEST_SPLIT,
        n_estimators=N_ESTIMATORS,
        random_state=RANDOM_STATE
    )
    
    results[name] = result
    mae_values.append(result['mae'])
    
    print(f"Training set shape: {result['train_shape']}")
    print(f"Test set shape: {result['test_shape']}")
    print(f"Model R² score: {result['model_score']:.4f}")
    print(f"Mean Absolute Error: {result['mae']:.4f}")
    print("-" * 60)

print(f"\nMAE results for all subgroups: {mae_values}")


Evaluating Females...
Training set shape: (334, 11891)
Test set shape: (63, 11891)
Model R² score: 0.0177
Mean Absolute Error: 1.4621
------------------------------------------------------------

Evaluating Males...
Training set shape: (313, 12692)
Test set shape: (84, 12692)
Model R² score: 0.0829
Mean Absolute Error: 1.3446
------------------------------------------------------------

Evaluating Whites...
Training set shape: (296, 12103)
Test set shape: (87, 12103)
Model R² score: 0.0709
Mean Absolute Error: 1.2120
------------------------------------------------------------

Evaluating Blacks...
Training set shape: (311, 9163)
Test set shape: (72, 9163)
Model R² score: -0.0225
Mean Absolute Error: 1.5711
------------------------------------------------------------

MAE results for all subgroups: [1.4620634920634925, 1.3446428571428573, 1.211954022988506, 1.5711111111111107]


## Database Export

Export stratified subgroups to MySQL database for further analysis.

In [ ]:
# Insert data into database tables
try:
    insert_data_to_db(stratified_male, TABLE_NAMES['male'], DB_CONFIG)
    insert_data_to_db(stratified_female, TABLE_NAMES['female'], DB_CONFIG)
    insert_data_to_db(stratified_black, TABLE_NAMES['black'], DB_CONFIG)
    insert_data_to_db(stratified_white, TABLE_NAMES['white'], DB_CONFIG)
    print("\nAll data exported to database successfully.")
except Exception as e:
    print(f"\nError during database export: {e}")